In [ ]:
import csv
import difflib
import gc
import os
import re
import time
from collections import defaultdict
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Machine Learning & Tensors
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

# Semantic Web / RDFLib
from rdflib import Graph, RDF, RDFS, URIRef, term
from rdflib.namespace import OWL, SKOS

# Prevent rdflib from raising an exception on invalid lexical forms
term._fail_on_invalid_lexical_form = False


class MOSAIC:
    _CAMEL_RE = re.compile(r'([a-z])([A-Z])')
    _PCT_RE = re.compile(r'%[0-9A-Fa-f]{2}')

    def __init__(self, model_name="BAAI/bge-m3", thresholds=None):
        self.thresholds = thresholds or {
            OWL.Class: 0.80,
            SKOS.Concept: 0.80,
            OWL.ObjectProperty: 0.88,
            OWL.DatatypeProperty: 0.88,
            OWL.NamedIndividual: 0.82
        }
        self.default_thres = 0.80
        self.model = None
        self.model_name = model_name

        # --- High-Performance Caches ---
        self.entity_cache = {}          # Key: Path -> Val: Extracted Entities dict
        self.string_emb_cache = {}      # Key: single string label -> Val: 1D Tensor (CPU)

        # --- CPU Thread Scaling Control ---
        cores = os.cpu_count() or 1
        if hasattr(os, 'sched_getaffinity'):
            try:
                cores = len(os.sched_getaffinity(0))
            except Exception:
                pass
        threads = max(4, cores - 1)
        torch.set_num_threads(threads)

    def init_model(self):
        if self.model is None:
            print(f" [MOSAIC] Initializing embedding engine on CPU: {self.model_name}")
            raw_model = SentenceTransformer(self.model_name, device="cpu")
            try:
                import torchao
                from torchao.quantization import quantize_
                from torchao.quantization.quant_api import int8_dynamic_activation_int8_weight
                print(" [MOSAIC] Applying modern torchao INT8 dynamic quantization.")
                quantize_(raw_model, int8_dynamic_activation_int8_weight())
                self.model = raw_model
            except (ImportError, Exception):
                try:
                    from torch.ao.quantization import quantize_dynamic
                    self.model = quantize_dynamic(raw_model, {torch.nn.Linear}, dtype=torch.qint8)
                except Exception:
                    self.model = raw_model

    def normalise_label(self, text: str) -> str:
        if not text:
            return ""
        text = self._CAMEL_RE.sub(r'\1 \2', str(text))
        text = text.replace('_', ' ').replace('-', ' ').lower().strip()
        # Aggressive truncation optimization: Keep only the first 20 words to prevent BGE-M3 thread bloat
        words = text.split()
        return " ".join(words[:20])

    def get_text_data(self, uri, graph) -> str:
        target_langs = ['de', 'fr', 'sl', 'hr', 'en', 'ar', 'es', 'it', 'nl']
        for lang in target_langs:
            for label in graph.objects(uri, SKOS.prefLabel):
                if hasattr(label, 'language') and label.language == lang:
                    return self.normalise_label(label)
            for label in graph.objects(uri, RDFS.label):
                if hasattr(label, 'language') and label.language == lang:
                    return self.normalise_label(label)

        label = graph.value(uri, RDFS.label) or graph.value(uri, SKOS.prefLabel)
        if label:
            return self.normalise_label(label)

        frag = str(uri).split('/')[-1].split('#')[-1]
        frag = self._PCT_RE.sub(' ', frag)
        return self.normalise_label(frag)

    def load_ontology(self, path: Path) -> Graph:
        g = Graph()
        formats = ["turtle", "xml"] if path.suffix == ".ttl" else (["xml", "turtle"] if path.suffix in [".owl", ".rdf", ".xml"] else [None])
        for fmt in formats:
            try:
                g.parse(str(path), format=fmt)
                return g
            except Exception:
                continue
        try:
            g.parse(str(path))
            return g
        except Exception as e:
            print(f" [MOSAIC] Failed to load {path.name}: {e}")
            return None

    def get_entities_cached(self, path: Path, graph: Graph):
        if path not in self.entity_cache:
            self.entity_cache[path] = self.extract_entities(graph)
        return self.entity_cache[path]

    def extract_entities(self, graph: Graph):
        skos_concepts = set(graph.subjects(RDF.type, SKOS.Concept))
        owl_classes = set(graph.subjects(RDF.type, OWL.Class)) - skos_concepts
        props = set(graph.subjects(RDF.type, OWL.ObjectProperty)).union(set(graph.subjects(RDF.type, OWL.DatatypeProperty)))
        all_subs = set(graph.subjects())
        insts = all_subs - skos_concepts - owl_classes - props

        insts = {i for i in insts if isinstance(i, URIRef) and "oboInOwl" not in str(i)}
        owl_classes = {c for c in owl_classes if isinstance(c, URIRef) and "oboInOwl" not in str(c)}
        skos_concepts = {c for c in skos_concepts if isinstance(c, URIRef) and "oboInOwl" not in str(c)}
        props = {p for p in props if isinstance(p, URIRef) and "oboInOwl" not in str(p)}

        entities = {}
        for entity_set, etype in [(owl_classes, OWL.Class), (skos_concepts, SKOS.Concept), (props, OWL.ObjectProperty), (insts, OWL.NamedIndividual)]:
            for s in entity_set:
                lbl_str = self.get_text_data(s, graph)
                entities[s] = {
                    "label": lbl_str,
                    "tokens": lbl_str.split(),
                    "type": etype,
                }
        return entities

    def get_embeddings_granular(self, labels: list, batch_size=512):
        """
        Guarantees a true 100% cache hit rate by fetching individual text fragments 
        instead of evaluating variable-length label lists.
        """
        missing_labels = [lbl for lbl in labels if lbl not in self.string_emb_cache]
        
        if missing_labels:
            self.init_model()
            encoded_missing = self.model.encode(
                missing_labels, convert_to_tensor=True, show_progress_bar=False, batch_size=batch_size
            )
            for lbl, tensor in zip(missing_labels, encoded_missing):
                self.string_emb_cache[lbl] = tensor.cpu()

        # Gather cached tensors and stack smoothly into a single operation block
        return torch.stack([self.string_emb_cache[lbl] for lbl in labels])

    def build_inverted_index(self, labels, tokens_list):
        index = defaultdict(list)
        counts = defaultdict(int)
        for tokens in tokens_list:
            for word in set(tokens):
                if len(word) > 2:
                    counts[word] += 1

        # Optimization: Cutoff aggressively at 1% of corpus size to avoid bloated maps
        max_limit = max(15, int(len(labels) * 0.01))
        for idx, tokens in enumerate(tokens_list):
            for word in tokens:
                if len(word) > 2 and counts[word] <= max_limit:
                    index[word].append(idx)
        return index

    def fast_token_match(self, src_tokens, tgt_labels, tgt_tokens_list, inverted_index):
        if not src_tokens: return 0.0, 0
        counts = defaultdict(int)
        for word in src_tokens:
            if word in inverted_index:
                for idx in inverted_index[word]:
                    counts[idx] += 1
        if not counts: return 0.0, 0

        best_idx = max(counts, key=counts.get)
        match_count = counts[best_idx]
        max_words = max(len(src_tokens), len(tgt_tokens_list[best_idx]))
        return (match_count / max_words if max_words > 0 else 0.0), best_idx

    def semantic_similarity_by_type(self, filtered_src, filtered_tgt, etype, batch_size=512, chunk_size=1024):
        src_subset = [ (uri, meta) for uri, meta in filtered_src.items() if meta["type"] == etype ]
        tgt_subset = [ (uri, meta) for uri, meta in filtered_tgt.items() if meta["type"] == etype ]
        if not src_subset or not tgt_subset:
            return []

        required_threshold = self.thresholds.get(etype, self.default_thres)

        src_uris, src_metas = zip(*src_subset)
        tgt_uris, tgt_metas = zip(*tgt_subset)

        src_labels = [m["label"] for m in src_metas]
        src_tokens_list = [m["tokens"] for m in src_metas]
        tgt_labels = [m["label"] for m in tgt_metas]
        tgt_tokens_list = [m["tokens"] for m in tgt_metas]

        tgt_index = self.build_inverted_index(tgt_labels, tgt_tokens_list)
        
        # Pull from granular, individual-label caching system
        with torch.inference_mode():
            emb1 = self.get_embeddings_granular(src_labels, batch_size=batch_size)
            emb2 = self.get_embeddings_granular(tgt_labels, batch_size=batch_size)

        n_src, n_tgt = len(src_labels), len(tgt_labels)
        use_blocking = n_tgt > 150  # Lowered barrier for earlier triggering of memory optimization
        candidates = []

        with torch.inference_mode():
            for start in range(0, n_src, chunk_size):
                end = min(start + chunk_size, n_src)
                sim_chunk = util.cos_sim(emb1[start:end], emb2)

                chunk_candidates = []
                if use_blocking:
                    for s_tokens in src_tokens_list[start:end]:
                        cols = set()
                        for w in s_tokens:
                            if w in tgt_index:
                                cols.update(tgt_index[w])
                        chunk_candidates.append(list(cols))

                for local_i in range(end - start):
                    i = start + local_i
                    s_lbl = src_labels[i]
                    s_tokens = src_tokens_list[i]

                    if use_blocking:
                        allowed_cols = chunk_candidates[local_i]
                        if not allowed_cols:
                            continue
                        row_sims = sim_chunk[local_i, allowed_cols]
                        if row_sims.numel() == 0:
                            continue
                        local_best = torch.argmax(row_sims).item()
                        best_tgt_idx = allowed_cols[local_best]
                        score = row_sims[local_best].item()
                    else:
                        score, best_tgt_idx = torch.max(sim_chunk[local_i], dim=0)
                        score = score.item()

                    t_lbl = tgt_labels[best_tgt_idx]

                    if use_blocking:
                        if abs(len(s_lbl) - len(t_lbl)) > max(len(s_lbl), len(t_lbl)) * 0.5:
                            ratio = 0.0
                        else:
                            ratio = difflib.SequenceMatcher(None, s_lbl, t_lbl).quick_ratio()
                            
                        if ratio < 0.45 and score < (required_threshold + 0.05):
                            score *= 0.75

                    if score < required_threshold:
                        t_score, fast_idx = self.fast_token_match(s_tokens, tgt_labels, tgt_tokens_list, tgt_index)
                        if t_score > 0.65 and t_score > score:
                            score = t_score
                            best_tgt_idx = fast_idx

                    if score >= required_threshold:
                        candidates.append({
                            "source": src_uris[i],
                            "target": tgt_uris[best_tgt_idx],
                            "type": etype,
                            "combined_score": score
                        })
        return candidates

    def align(self, src_graph: Graph, tgt_graph: Graph, src_path: Path, tgt_path: Path, preferred_skos_pred: str = "http://www.w3.org/2002/07/owl#sameAs"):
        src_ents = self.get_entities_cached(src_path, src_graph)
        tgt_ents = self.get_entities_cached(tgt_path, tgt_graph)

        final_pool = []
        claimed_src = set()
        claimed_tgt = set()

        tgt_lookup = {meta["label"]: uri for uri, meta in tgt_ents.items() if meta["label"]}

        for s_uri, s_meta in src_ents.items():
            s_lbl = s_meta["label"]
            if s_lbl in tgt_lookup:
                t_uri = tgt_lookup[s_lbl]
                t_meta = tgt_ents[t_uri]
                if s_meta["type"] == t_meta["type"]:
                    claimed_src.add(s_uri)
                    claimed_tgt.add(t_uri)
                    final_pool.append({
                        "source": s_uri,
                        "target": t_uri,
                        "type": s_meta["type"],
                        "combined_score": 1.0
                    })

        filtered_src = {k: v for k, v in src_ents.items() if k not in claimed_src}
        filtered_tgt = {k: v for k, v in tgt_ents.items() if k not in claimed_tgt}

        if filtered_src and filtered_tgt:
            distinct_types = [OWL.Class, SKOS.Concept, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.NamedIndividual]
            sem_candidates = []
            for etype in distinct_types:
                sem_candidates.extend(self.semantic_similarity_by_type(filtered_src, filtered_tgt, etype))

            sorted_pairs = sorted(sem_candidates, key=lambda x: x["combined_score"], reverse=True)
            for c in sorted_pairs:
                if c["source"] not in claimed_src and c["target"] not in claimed_tgt:
                    claimed_src.add(c["source"])
                    claimed_tgt.add(c["target"])
                    final_pool.append(c)

        alignments = set()
        eq_class = "http://www.w3.org/2002/07/owl#equivalentClass"
        eq_prop = "http://www.w3.org/2002/07/owl#equivalentProperty"
        same_as = "http://www.w3.org/2002/07/owl#sameAs"

        for c in final_pool:
            s_uri, t_uri, etype = c["source"], c["target"], c["type"]
            if etype == OWL.Class:
                alignments.add((str(s_uri), eq_class, str(t_uri)))
            elif etype == SKOS.Concept:
                alignments.add((str(s_uri), preferred_skos_pred, str(t_uri)))
            elif etype in [OWL.ObjectProperty, OWL.DatatypeProperty]:
                alignments.add((str(s_uri), eq_prop, str(t_uri)))
            else:
                alignments.add((str(s_uri), same_as, str(t_uri)))

        return alignments


class OAEITrackRunner:

    def __init__(self, matcher: MOSAIC):
        self.matcher = matcher
        self.log = []

    def load_reference_alignments(self, path: Path) -> set:
        ref_set = set()
        g = Graph()
        try:
            g.parse(str(path), format="turtle")
            valid_preds = {
                "http://www.w3.org/2002/07/owl#equivalentClass",
                "http://www.w3.org/2000/01/rdf-schema#subClassOf",
                "http://www.w3.org/2002/07/owl#equivalentProperty",
                "http://www.w3.org/2000/01/rdf-schema#subPropertyOf",
                "http://www.w3.org/2002/07/owl#sameAs",
            }
            for s, p, o in g:
                if str(p) in valid_preds:
                    nodes = sorted([str(s), str(o)])
                    ref_set.add((nodes[0], str(p), nodes[1]))
        except Exception as e:
            print(f" Could not read reference file {path.name}: {e}")
        return ref_set

    def serialize_alignments_to_ttl(self, alignments: set, path: Path):
        g = Graph()
        for src, pred, tgt in alignments:
            g.add((URIRef(src), URIRef(pred), URIRef(tgt)))
        try:
            g.serialize(destination=str(path), format="turtle")
            print(f"   [MOSAIC] Output saved to: {path.parent.name}/{path.name}")
        except Exception as e:
            print(f"   [MOSAIC] Serialization error: {e}")

    def calculate_metrics(self, sys_align, ref_align):
        if not ref_align:
            return 0.0, 0.0, 0.0
        sys_canon = set()
        for s, p, o in sys_align:
            nodes = sorted([str(s), str(o)])
            sys_canon.add((nodes[0], str(p), nodes[1]))

        tp = len(sys_canon.intersection(ref_align))
        p = tp / len(sys_canon) if sys_canon else 0.0
        r = tp / len(ref_align) if ref_align else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        return round(p, 4), round(r, 4), round(f1, 4)

    def find_ontology_file(self, folder: Path, name: str) -> Path:
        for ext in [".owl", ".rdf", ".ttl", ".xml"]:
            p = folder / f"{name}{ext}"
            if p.exists():
                return p
        return None

    def run_all_tracks(self, base_dir: str, csv_out: str = "mosaic_evaluation_report.csv"):
        start_global = time.time()
        base_path = Path(base_dir)
        res_dir = Path("../results")
        res_dir.mkdir(parents=True, exist_ok=True)

        if not base_path.exists():
            print(f"Error: Base directory '{base_dir}' does not exist.")
            return

        for track in base_path.iterdir():
            if not track.is_dir():
                continue

            print(f"\n" + "=" * 50)
            print(f" TRACK RUNNER: {track.name.upper()}")
            print(f"=" * 50)

            tasks = list(track.glob("*.ttl"))
            p_sum, r_sum, f_sum, t_sum = 0.0, 0.0, 0.0, 0.0
            count = 0

            for tf in tasks:
                parts = tf.stem.split("-")
                if len(parts) != 2:
                    if "human-mouse" in tf.stem:
                        parts = ["human", "mouse"]
                    else:
                        continue

                ont_folder = track / "ontologies"
                src_p = self.find_ontology_file(ont_folder, parts[0])
                tgt_p = self.find_ontology_file(ont_folder, parts[1])

                print(f"\nMOSAIC Task: {parts[0]} ➔ {parts[1]}")

                if not src_p or not tgt_p:
                    print(" Skipping task. Missing ontology file.")
                    continue

                ref_align = self.load_reference_alignments(tf)
                preferred_skos_pred = "http://www.w3.org/2002/07/owl#sameAs"
                for _, p, _ in ref_align:
                    if "equivalentClass" in p:
                        preferred_skos_pred = "http://www.w3.org/2002/07/owl#equivalentClass"
                        break

                with ThreadPoolExecutor(max_workers=2) as executor:
                    future_src = executor.submit(self.matcher.load_ontology, src_p)
                    future_tgt = executor.submit(self.matcher.load_ontology, tgt_p)
                    src_g = future_src.result()
                    tgt_g = future_tgt.result()

                if src_g and tgt_g:
                    t0 = time.time()
                    alignments = self.matcher.align(src_g, tgt_g, src_p, tgt_p, preferred_skos_pred=preferred_skos_pred)
                    dt = round(time.time() - t0, 2)

                    print(f" Step complete. MOSAIC returned {len(alignments)} matches in {dt}s.")

                    out_ttl = res_dir / f"mosaic_{track.name}_{tf.name}"
                    self.serialize_alignments_to_ttl(alignments, out_ttl)

                    p, r, f1 = self.calculate_metrics(alignments, ref_align)
                    print(f"   Metrics -> Precision: {p}, Recall: {r}, F1-Score: {f1} (Time: {dt}s)")

                    self.log.append({
                        "Track": track.name,
                        "Task": tf.stem,
                        "Precision": p,
                        "Recall": r,
                        "F1-Score": f1,
                        "Time (s)": dt,
                        "Type": "Task",
                    })

                    p_sum += p
                    r_sum += r
                    f_sum += f1
                    t_sum += dt
                    count += 1

                    del src_g, tgt_g
                    gc.collect()

            if count > 0:
                avg_p = round(p_sum / count, 4)
                avg_r = round(r_sum / count, 4)
                avg_f1 = round(f_sum / count, 4)
                avg_t = round(t_sum / count, 2)

                print(f"\n Track [{track.name}] AVERAGES -> P: {avg_p}, R: {avg_r}, F1: {avg_f1} | Avg Time: {avg_t}s")

                self.log.append({
                    "Track": track.name,
                    "Task": "TRACK_AVERAGE",
                    "Precision": avg_p,
                    "Recall": avg_r,
                    "F1-Score": avg_f1,
                    "Time (s)": avg_t,
                    "Type": "Average",
                })

        total_runtime = round(time.time() - start_global, 2)
        print(f"\n" + "=" * 50)
        print(f" RUN COMPLETION: Finished in {total_runtime}s.")
        print(f"=" * 50)

        self.log.append({
            "Track": "ALL_TRACKS",
            "Task": "TOTAL_PROGRAM_TIME",
            "Precision": "",
            "Recall": "",
            "F1-Score": "",
            "Time (s)": total_runtime,
            "Type": "Summary",
        })
        self.results_to_csv(csv_out)

    def results_to_csv(self, filename: str):
        fields = ["Track", "Task", "Precision", "Recall", "F1-Score", "Time (s)", "Type"]
        with open(filename, mode="w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(self.log)
        print(f" Compilation written to: {filename}")


if __name__ == "__main__":
    custom_thresholds = {
        OWL.Class: 0.78,
        SKOS.Concept: 0.80,
        OWL.ObjectProperty: 0.87,
        OWL.DatatypeProperty: 0.87,
        OWL.NamedIndividual: 0.82
    }

    m = MOSAIC(model_name="BAAI/bge-m3", thresholds=custom_thresholds)
    runner = OAEITrackRunner(matcher=m)
    runner.run_all_tracks("../tracks", csv_out="mosaic_evaluation_report.csv")

c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0703 19:26:48.096000 22980 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0703 19:26:48.174000 22980 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.



 TRACK RUNNER: ANATOMY

MOSAIC Task: human ➔ mouse
 [MOSAIC] Initializing embedding engine on CPU: BAAI/bge-m3


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 55859.29it/s]
C:\Users\PC\AppData\Local\Temp\ipykernel_22980\1482467862.py:68: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  self.model = quantize_dynamic(raw_model, {torch.nn.Linear}, dtype=torch.qint8)


 Step complete. MOSAIC returned 1463 matches in 126.5s.
   [MOSAIC] Output saved to: results/mosaic_anatomy_human-mouse.ttl
   Metrics -> Precision: 0.8462, Recall: 0.8166, F1-Score: 0.8312 (Time: 126.5s)

 Track [anatomy] AVERAGES -> P: 0.8462, R: 0.8166, F1: 0.8312 | Avg Time: 126.5s

 TRACK RUNNER: BIO-ML

MOSAIC Task: ncit ➔ doid


KeyboardInterrupt: 